# 3. OpenFDA Inference Pipeline

**Purpose**: Apply trained model to OpenFDA NDC drugs data, focusing on most frequently appearing organizations.

## Sections
1. Load config and saved model
2. Query top N most frequent organizations
3. Process with progress tracking
4. Cluster predictions
5. Export results


---
## 1. Setup and Load Model


In [1]:
import sys
sys.path.insert(0, '/Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test')

import pandas as pd
import numpy as np
import json
from pathlib import Path

# Splink imports
from splink import Linker, DuckDBAPI

# Local imports
from config import config
from utils import (
    DatabaseManager, 
    log_step, 
    Timer,
    ProgressTracker,
    save_checkpoint,
    load_checkpoint,
    load_json
)
from data_prep import (
    load_mismatched,
    load_dim_org,
    create_unified_schema,
    filter_bad_records,
    create_blocking_keys,
    add_distinctive_tokens,  # Changed from add_idf_based_features
    add_token_set_features,
    compute_token_statistics,
    get_corpus_stopwords
)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

print("Imports loaded successfully")


Imports loaded successfully


In [2]:
import sys

import pandas as pd
import numpy as np
import json
from pathlib import Path

# Splink imports
from splink import Linker, DuckDBAPI

# Local imports
from config import config
from utils import (
    DatabaseManager, 
    log_step, 
    Timer,
    ProgressTracker,
    save_checkpoint,
    load_checkpoint,
    load_json
)
from data_prep import (
    load_mismatched,
    load_dim_org,
    create_unified_schema,
    filter_bad_records,
    create_blocking_keys,
    add_distinctive_tokens,
    add_token_set_features,
    compute_token_statistics,
    get_corpus_stopwords
)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

print("Imports loaded successfully")


Imports loaded successfully


In [3]:
# Initialize database
db = DatabaseManager()

# Load saved model
model_path = config.paths.MODEL_FILE
if not Path(model_path).exists():
    raise FileNotFoundError(f"Model not found at {model_path}. Run 2_training.ipynb first.")

model_json = load_json(model_path, "Trained model")

print(f"\nMODEL LOADED")
print("=" * 50)
print(f"Path: {model_path}")
if 'training_metadata' in model_json:
    meta = model_json['training_metadata']
    print(f"Trained on: {meta.get('timestamp', 'N/A')}")
    print(f"Training records: {meta.get('dim_org_records', 0):,} dim_org + {meta.get('grid_records', 0):,} GRID")


[17:15:35]  Loading JSON: Trained model

MODEL LOADED
Path: /Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/models/model_v1.json
Trained on: 2026-01-20T14:32:10.607781
Training records: 109,851 dim_org + 109,856 GRID


In [4]:
# Load reference data (dim_org) for linking
# Using cached data if available
cached_dim_org = load_checkpoint(config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org cache")

if cached_dim_org is not None:
    dim_org_df = cached_dim_org
    print(f"Loaded cached dim_org: {len(dim_org_df):,} records")
else:
    dim_org_raw = load_dim_org(db, sample_size=None)  # Full load for inference
    dim_org_df = create_unified_schema(dim_org_raw, 'dim_org')
    dim_org_df, _ = filter_bad_records(dim_org_df)
    dim_org_df = create_blocking_keys(dim_org_df)
    print(f"Loaded dim_org: {len(dim_org_df):,} records")


[17:15:36]  Loading checkpoint: dim_org cache
[17:15:36]    Loaded 109,851 rows
Loaded cached dim_org: 109,851 records


---
## 2. Query Top N Most Frequent OpenFDA Organizations


In [5]:
# Configuration
TOP_N_ORGS = 10000  # Number of most frequent organizations to process

# Query to find most frequent organizations in OpenFDA
top_orgs_query = f"""
    SELECT 
        name,
        COUNT(*) as frequency
    FROM allsci_prod_gold.potential_mismatched_organizations
    WHERE source_table = 'open_fda_silver.ndc_drugs'
    GROUP BY name
    ORDER BY frequency DESC
    LIMIT {TOP_N_ORGS}
"""

top_orgs_df = db.execute_query(top_orgs_query, f"Top {TOP_N_ORGS} OpenFDA organizations")

print(f"\nTOP {TOP_N_ORGS} MOST FREQUENT OPENFDA ORGANIZATIONS")
print("=" * 60)
print(f"Total unique orgs queried: {len(top_orgs_df):,}")
print(f"Total records covered: {top_orgs_df['frequency'].sum():,}")
print(f"\nTop 10 by frequency:")
print(top_orgs_df.head(10).to_string())
print(f"\nFrequency range: {top_orgs_df['frequency'].min():,} - {top_orgs_df['frequency'].max():,}")


[17:15:39]  Executing: Top 10000 OpenFDA organizations
[17:15:47]    Returned 10,000 rows in 8.2s

TOP 10000 MOST FREQUENT OPENFDA ORGANIZATIONS
Total unique orgs queried: 10,000
Total records covered: 298,160

Top 10 by frequency:
                                         name  frequency
0                        Bryant Ranch Prepack       9478
1                Hahnemann Laboratories, INC.       8799
2                    A-S Medication Solutions       5779
3                           REMEDYREPACK INC.       5107
4                                      Boiron       4211
5                            Proficient Rx LP       3676
6                    Aurobindo Pharma Limited       3055
7                    Greer Laboratories, Inc.       2654
8                 PD-Rx Pharmaceuticals, Inc.       2456
9  The Procter & Gamble Manufacturing Company       2263

Frequency range: 1 - 9,478


In [6]:
# Compute IDF scores from reference data
idf_scores = compute_token_statistics(dim_org_df, 'name_normalized')
corpus_stopwords = get_corpus_stopwords(idf_scores, percentile=0.10)

print(f"\nIDF SCORES COMPUTED")
print("=" * 50)
print(f"Vocabulary size: {len(idf_scores):,} tokens")
print(f"Corpus stopwords: {len(corpus_stopwords):,}")


[17:15:47]  Computing token IDF statistics...
[17:15:47]    Computed IDF for 64,820 unique tokens
[17:15:47]    IDF range: 1.90 (most common) to 11.61 (most rare)
[17:15:47]    Auto-identified 6786 corpus stopwords (IDF <= 10.00)

IDF SCORES COMPUTED
Vocabulary size: 64,820 tokens
Corpus stopwords: 6,786


---
## 3. Load and Process Top Organizations


In [7]:
# Load one record per org for top N organizations from OpenFDA
# Uses ROW_NUMBER() instead of DISTINCT ON (which is PostgreSQL-only)

filtered_query = f"""
WITH top_orgs AS (
    SELECT name, COUNT(*) as frequency
    FROM allsci_prod_gold.potential_mismatched_organizations
    WHERE source_table = 'open_fda_silver.ndc_drugs'
    GROUP BY name
    ORDER BY frequency DESC
    LIMIT {TOP_N_ORGS}
),
one_per_org AS (
    SELECT m.*, 
           ROW_NUMBER() OVER (PARTITION BY m.name ORDER BY m.mismatch_id) as rn
    FROM allsci_prod_gold.potential_mismatched_organizations m
    INNER JOIN top_orgs t ON m.name = t.name
    WHERE m.source_table = 'open_fda_silver.ndc_drugs'
)
SELECT * FROM one_per_org WHERE rn = 1
"""

log_step(f"Loading top {TOP_N_ORGS:,} organizations from OpenFDA (1 record each)...")
openfda_df = db.execute_query(filtered_query, "OpenFDA top organizations")

# Drop the rn column we used for deduplication
if 'rn' in openfda_df.columns:
    openfda_df = openfda_df.drop(columns=['rn'])

print(f"\nLOADED OPENFDA DATA")
print("=" * 50)
print(f"Records loaded: {len(openfda_df):,}")
print(f"Unique organizations: {openfda_df['name'].nunique():,}")

[17:15:47]  Loading top 10,000 organizations from OpenFDA (1 record each)...
[17:15:47]  Executing: OpenFDA top organizations
[17:15:58]    Returned 10,000 rows in 10.1s

LOADED OPENFDA DATA
Records loaded: 10,000
Unique organizations: 10,000


In [8]:
import re

def parse_metadata_string(metadata_str):
    """Parse Athena map string like {key1=value1, key2=value2} into dict"""
    if pd.isna(metadata_str) or not metadata_str:
        return {}
    
    # Remove outer braces
    content = str(metadata_str).strip('{}')
    
    result = {}
    # Split by ", " but handle nested values
    current_key = None
    current_value = []
    
    for part in re.split(r',\s*(?=[a-z_]+=)', content):
        if '=' in part:
            if current_key:
                result[current_key] = ''.join(current_value).strip()
            key_val = part.split('=', 1)
            current_key = key_val[0].strip()
            current_value = [key_val[1]] if len(key_val) > 1 else []
        else:
            current_value.append(', ' + part)
    
    if current_key:
        result[current_key] = ''.join(current_value).strip()
    
    return result

# Parse metadata into separate columns
print("Parsing metadata...")
metadata_parsed = openfda_df['metadata'].apply(parse_metadata_string)
metadata_df = pd.DataFrame(metadata_parsed.tolist())

# Combine with original columns
openfda_df = pd.concat([openfda_df.drop(columns=['metadata']), metadata_df], axis=1)

print(f"Parsed columns: {openfda_df.columns.tolist()}")
print(f"Records: {len(openfda_df):,}")

Parsing metadata...
Parsed columns: ['mismatch_id', 'name', 'source_table', 'source_entity_id', 'ingestion_datetime', 'name_clean', 'product_ndc', 'spl_id', 'relationship_type', 'drug_generic_name', 'name_normalized', 'drug_brand_name']
Records: 10,000


In [9]:
# Prepare data for Splink
source_id_mapping_raw = openfda_df[['mismatch_id', 'source_entity_id']].copy()

# Add missing columns that create_unified_schema expects
missing_cols = ['country', 'country_code', 'city', 'state', 'latitude', 'longitude', 'name_aliases', 'type', 'name_prefix_5', 'name_prefix_10']
for col in missing_cols:
    if col not in openfda_df.columns:
        openfda_df[col] = None

# Create unified schema
chunk_unified = create_unified_schema(openfda_df, 'mismatched')

# Extract mismatch_id from unique_id
chunk_unified['mismatch_id'] = chunk_unified['unique_id'].str.replace('mis_', '', regex=False)

# Merge source_entity_id back
chunk_unified = chunk_unified.merge(source_id_mapping_raw, on='mismatch_id', how='left', suffixes=('_drop', ''))

# Clean up duplicate columns
drop_cols = [c for c in chunk_unified.columns if c.endswith('_drop')]
chunk_unified = chunk_unified.drop(columns=drop_cols, errors='ignore')

# Keep original variable name for later cells
source_id_mapping = chunk_unified[['unique_id', 'source_entity_id']].copy()
print(f"source_entity_id mapped: {source_id_mapping['source_entity_id'].notna().sum():,} records")

# Filter bad records
chunk_filtered, removed = filter_bad_records(chunk_unified)
print(f"Filtered: {len(removed):,} bad records")

# Create blocking keys
chunk_prepared = create_blocking_keys(chunk_filtered)

# Add distinctive tokens
chunk_prepared = add_distinctive_tokens(chunk_prepared, idf_scores, corpus_stopwords)

# Add token set features
chunk_prepared = add_token_set_features(chunk_prepared)

print(f"\nPREPARED DATA")
print("=" * 50)
print(f"Records ready for inference: {len(chunk_prepared):,}")

[17:15:58]  Creating unified schema for mismatched...
[17:15:58]    Created unified schema: 10,000 rows, 18 columns
source_entity_id mapped: 10,000 records
[17:15:58]  Filtering bad records...
[17:15:58]    Removed 11 of 10,000 records (0.1%)
[17:15:58]      - short_name: 11
Filtered: 11 bad records
[17:15:58]  Creating blocking keys...
[17:15:58]  Adding phonetic codes...
[17:15:58]    Phonetic coverage: 100.0%
[17:15:58]    Added blocking keys to 9,989 records
[17:15:58]  Adding distinctive tokens...
[17:15:58]    Loaded 34,655 geographic stopwords
[17:15:58]    Geographic stopwords: 34,655 (locations filtered out)
[17:15:58]    Distinctive token coverage: 100.0%
[17:15:58]  Adding token set features for containment detection...
[17:15:58]    Average token count: 2.6

PREPARED DATA
Records ready for inference: 9,989


In [10]:
# Ensure dim_org has distinctive token features
if 'distinctive_token' not in dim_org_df.columns:
    print("Adding distinctive tokens to dim_org...")
    dim_org_df = add_distinctive_tokens(dim_org_df, idf_scores, corpus_stopwords)
    dim_org_df = add_token_set_features(dim_org_df)


Adding distinctive tokens to dim_org...
[17:15:58]  Adding distinctive tokens...
[17:15:58]    Geographic stopwords: 34,655 (locations filtered out)
[17:15:59]    Distinctive token coverage: 99.9%
[17:15:59]  Adding token set features for containment detection...
[17:15:59]    Average token count: 3.2


In [11]:
# Run inference
log_step("Running Splink inference...")

# Remove training metadata (not a Splink setting)
model_settings = {k: v for k, v in model_json.items() if k != 'training_metadata'}

# Override blocking rules for geo-sparse inference data (OpenFDA lacks country/city)
inference_blocking_rules = [
    {"blocking_rule": "l.name_normalized = r.name_normalized", "sql_dialect": "duckdb"},
    {"blocking_rule": "l.distinctive_token = r.distinctive_token", "sql_dialect": "duckdb"},
    {"blocking_rule": "l.name_metaphone = r.name_metaphone", "sql_dialect": "duckdb"},
    {"blocking_rule": "substr(l.name_normalized, 1, 10) = substr(r.name_normalized, 1, 10)", "sql_dialect": "duckdb"},
]
model_settings['blocking_rules_to_generate_predictions'] = inference_blocking_rules
print(f"Using {len(inference_blocking_rules)} inference blocking rules (geo-agnostic)")

# Initialize linker
linker = Linker(
    [dim_org_df, chunk_prepared],
    model_settings,
    db_api=DuckDBAPI()
)

# Predict
predictions = linker.inference.predict(
    threshold_match_probability=config.matching.THRESHOLD_PREDICTION
)
predictions_df = predictions.as_pandas_dataframe()
predictions_df['source_table'] = 'open_fda_silver.ndc_drugs'

print(f"\nINFERENCE COMPLETE")
print("=" * 50)
print(f"Total predictions: {len(predictions_df):,}")

[17:15:59]  Running Splink inference...
Using 4 inference blocking rules (geo-agnostic)


Blocking time: 0.15 seconds
Predict time: 2.07 seconds



INFERENCE COMPLETE
Total predictions: 65,805


In [12]:
# Join source_entity_id back
combined_predictions = predictions_df.merge(
    source_id_mapping,
    left_on='unique_id_r',
    right_on='unique_id',
    how='left'
).drop(columns=['unique_id'], errors='ignore')

print(f"Joined source_entity_id: {combined_predictions['source_entity_id'].notna().sum():,} records")

Joined source_entity_id: 65,805 records


In [13]:
# Analyze unmatched organizations
matched_org_ids = set(combined_predictions['unique_id_r'].unique())
all_org_ids = set(chunk_prepared['unique_id'].unique())
unmatched_ids = all_org_ids - matched_org_ids

print(f"\nUNMATCHED ORGANIZATIONS ANALYSIS")
print("=" * 60)
print(f"Total organizations: {len(all_org_ids):,}")
print(f"Matched: {len(matched_org_ids):,} ({100*len(matched_org_ids)/len(all_org_ids):.1f}%)")
print(f"Unmatched: {len(unmatched_ids):,} ({100*len(unmatched_ids)/len(all_org_ids):.1f}%)")

# Show sample unmatched orgs
unmatched_df = chunk_prepared[chunk_prepared['unique_id'].isin(unmatched_ids)].copy()
if len(unmatched_df) > 0:
    print(f"\nSample unmatched organizations:")
    print(unmatched_df[['name', 'name_normalized', 'distinctive_token']].head(20).to_string())
    
    # Save for review
    save_checkpoint(unmatched_df, config.paths.DATA_DIR + "/openfda_no_match.parquet", "OpenFDA unmatched")


UNMATCHED ORGANIZATIONS ANALYSIS
Total organizations: 9,989
Matched: 5,054 (50.6%)
Unmatched: 4,935 (49.4%)

Sample unmatched organizations:
                                               name                        name_normalized      distinctive_token
2                                       Nufabrx LLC                                nufabrx                nufabrx
4                              Pineapple Brands LLC                       pineapple brands              pineapple
9                          Nenningers Naturals, LLC                    nenningers naturals             nenningers
10                                   Diversey, Inc.                               diversey               diversey
11                       Softgel Healthcare Pvt Ltd                 softgel healthcare pvt                softgel
13                                   PADAGIS US LLC                             padagis us                padagis
14                                      Kaleo, Inc.         

In [14]:
# Disambiguate many-to-many matches
print("DISAMBIGUATING MANY-TO-MANY MATCHES")
print("=" * 50)

dupes_per_mismatched = combined_predictions.groupby('unique_id_r').size()
multi_match_count = (dupes_per_mismatched > 1).sum()
print(f"Mismatched records with multiple matches: {multi_match_count:,}")

if multi_match_count > 0:
    def add_tiebreaker_score(df):
        df = df.copy()
        if 'country_code_r' in df.columns:
            df['country_bonus'] = (df['country_code_l'] == df['country_code_r']).astype(int)
        else:
            df['country_bonus'] = 0
        df['name_len'] = df['name_l'].str.len() if 'name_l' in df.columns else df['name_normalized_l'].str.len()
        return df

    combined_predictions = add_tiebreaker_score(combined_predictions)
    combined_predictions_sorted = combined_predictions.sort_values(
        by=['unique_id_r', 'match_probability', 'country_bonus', 'name_len'],
        ascending=[True, False, False, True]
    )
    disambiguated = combined_predictions_sorted.groupby('unique_id_r').first().reset_index()

    print(f"\nBefore disambiguation: {len(combined_predictions):,}")
    print(f"After disambiguation:  {len(disambiguated):,}")
    
    save_checkpoint(combined_predictions, config.paths.DATA_DIR + "/openfda_all_matches.parquet", "All OpenFDA matches")
    combined_predictions = disambiguated
    combined_predictions = combined_predictions.drop(columns=['country_bonus', 'name_len'], errors='ignore')
else:
    print("No duplicate matches found")


DISAMBIGUATING MANY-TO-MANY MATCHES
Mismatched records with multiple matches: 3,716

Before disambiguation: 65,805
After disambiguation:  5,054
[17:16:03]  Saving checkpoint: All OpenFDA matches
[17:16:03]    Saved 65,805 rows to /Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/data/openfda_all_matches.parquet


---
## 4. Cluster Predictions


In [43]:
combined_predictions

,unique_id_r,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,name_normalized_l,name_normalized_r,gamma_name_normalized,distinctive_soundex_l,distinctive_soundex_r,gamma_phonetic_match,distinctive_tokens_l,distinctive_tokens_r,gamma_distinctive_match,gamma_token_overlap,name_tokens_l,name_tokens_r,gamma_containment_check,distinctive_token_l,distinctive_token_r,gamma_first_token_match,all_names_l,all_names_r,name_l,name_r,gamma_alias_match,country_code_l,country_code_r,gamma_country_match,city_l,city_r,gamma_city_match,name_metaphone_l,name_metaphone_r,match_key,source_table,source_entity_id,cluster_id,rolled_up_from,rolled_up_from_name,confidence_tier
0,mis_0000d4543ed48d42e5b74b97275e2544,23.076170,1.000000,__splink__input_table_0,__splink__input_table_1,dim_ASC-OR-0000000033155-1.0-1724880247,unichem,unichem pharmaceuticals,1,U525,U525,1,[unichem],"[unichem, pharmaceuticals]",1,1,[unichem],"[pharmaceuticals, unichem]",3,unichem,unichem,3,"[Unichem (Slovenia), [Unichem (Slovenia)]]","[Unichem Pharmaceuticals (USA), Inc.]",Unichem (Slovenia),"Unichem Pharmaceuticals (USA), Inc.",0,SI,None,-1,Vrhnika,None,-1,UNXM,UNXM,1,open_fda_silver.ndc_drugs,29300-129_04e2c83b-4c31-41f8-bc75-f82ef1b30d07,0.0,NaN,NaN,0.95-1.0 (Very High)
1,mis_0001633fb26f5714058707a712217236,23.772254,1.000000,__splink__input_table_0,__splink__input_table_1,dim_ASC-OR-0000000107713-1.0-1724880264,novartis,sandoz,2,S532,S532,1,[sandoz],[sandoz],1,1,"[group, sandoz]",[sandoz],3,sandoz,sandoz,3,"[[Novartis (Switzerland)], Novartis (Switzerland)]",[Sandoz Inc],Novartis (Switzerland),Sandoz Inc,0,CH,None,-1,Basel,None,-1,SNTS,SNTS,1,open_fda_silver.ndc_drugs,0781-7104_eae28053-9c1d-4779-b322-6ed262879640,1.0,dim_ASC-OR-0000000057821-1.0-1724880251,Sandoz Group AG (Switzerland),0.95-1.0 (Very High)
2,mis_0001a660e01aff71810545216bcf8c4e,26.244397,1.000000,__splink__input_table_0,__splink__input_table_1,dim_ASC-OR-0000000079092-1.0-1724880257,hahnemann university hospital,hahnemann laboratories,1,H555,H555,1,[hahnemann],"[hahnemann, laboratories]",1,1,"[hahnemann, hospital, university]","[hahnemann, laboratories]",2,hahnemann,hahnemann,3,"[Hahnemann University Hospital, [Hahnemann University Hospital]]","[Hahnemann Laboratories, INC.]",Hahnemann University Hospital,"Hahnemann Laboratories, INC.",0,US,None,-1,Philadelphia,None,-1,HNMN,HNMN,1,open_fda_silver.ndc_drugs,37662-3228_fc636b53-c038-2fef-e053-6394a90aaca6,2.0,NaN,NaN,0.95-1.0 (Very High)
3,mis_0001b14ac160f9d4b7deaff6066c4195,17.749490,0.999995,__splink__input_table_0,__splink__input_table_1,dim_ASC-OR-0000000075192-1.0-1724880256,universitas tri dharma,dharma research,0,D650,D650,1,"[dharma, universitas]",[dharma],1,1,"[dharma, tri, universitas]","[dharma, research]",2,dharma,dharma,3,"[[Universitas Tri Dharma], Universitas Tri Dharma]","[Dharma Research, Inc.]",Universitas Tri Dharma,"Dharma Research, Inc.",0,ID,None,-1,Balikpapan,None,-1,UNFRSTS,THRM,1,open_fda_silver.ndc_drugs,53045-271_91060eb1-2d2e-4100-912f-05e7b46c9ae3,3.0,NaN,NaN,0.95-1.0 (Very High)
4,mis_0001cd650d25d8460322bda8317c2c1d,23.076170,1.000000,__splink__input_table_0,__splink__input_table_1,dim_ASC-OR-0000000056928-1.0-1724880250,pbr laboratories,greer laboratories,1,L163,L163,1,[laboratories],[laboratories],1,1,"[laboratories, pbr]","[greer, laboratories]",3,laboratories,laboratories,3,"[PBR Laboratories, [PBR Laboratories]]","[Greer Laboratories, Inc.]",PBR Laboratories,"Greer Laboratories, Inc.",0,CA,None,-1,Edmonton,None,-1,PBR,KRR,1,open_fda_silver.ndc_drugs,22840-9309_36ac4a35-6d7b-a249-e063-6294a90a0b70,4.0,NaN,NaN,0.95-1.0 (Very High)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5049,mis_f75de9699da72b22e83bebe716494da1,26.244397,1.000000,__splink__input_table_0,__splink__input_table_1,dim_ASC-OR-0000000052117-1.0-1724880249,leading age,leading edge innovations,1,L352,L352,

In [15]:
if len(combined_predictions) > 0:
    print("CLUSTERING PREDICTIONS")
    print("=" * 50)
    
    import networkx as nx
    
    high_conf = combined_predictions[
        combined_predictions['match_probability'] >= config.matching.CLUSTER_THRESHOLD
    ]
    
    print(f"Building graph from {len(high_conf):,} edges above threshold {config.matching.CLUSTER_THRESHOLD}")
    
    G = nx.Graph()
    for _, row in high_conf.iterrows():
        G.add_edge(row['unique_id_l'], row['unique_id_r'], weight=row['match_probability'])
    
    components = list(nx.connected_components(G))
    
    cluster_records = []
    for cluster_id, members in enumerate(components):
        for unique_id in members:
            cluster_records.append({'unique_id': unique_id, 'cluster_id': cluster_id})
    
    clusters_df = pd.DataFrame(cluster_records)
    cluster_sizes = clusters_df.groupby('cluster_id').size()
    
    print(f"\nTotal clusters: {len(cluster_sizes):,}")
    print(f"Largest cluster: {cluster_sizes.max()} records")
    
    combined_predictions = combined_predictions.merge(
        clusters_df.rename(columns={'unique_id': 'unique_id_l'}),
        on='unique_id_l',
        how='left'
    )
    
    save_checkpoint(clusters_df, config.paths.DATA_DIR + "/openfda_clusters.parquet", "OpenFDA clusters")


CLUSTERING PREDICTIONS
Building graph from 4,773 edges above threshold 0.85

Total clusters: 2,191
Largest cluster: 65 records
[17:16:03]  Saving checkpoint: OpenFDA clusters
[17:16:03]    Saved 6,964 rows to /Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/data/openfda_clusters.parquet


---
## 5. Hierarchy Roll-up


In [16]:
print("LOADING ORGANIZATION HIERARCHY")
print("=" * 50)

hierarchy_query = """
SELECT 
    to_organization_allsci_id as child_id,
    from_organization_allsci_id as parent_id
FROM allsci_prod_gold.fact_organization_hierarchy_organization
WHERE relationship_type = 'parent'
"""

hierarchy_df = db.execute_query(hierarchy_query, "Organization hierarchy")
print(f"Loaded {len(hierarchy_df):,} parent-child relationships")


LOADING ORGANIZATION HIERARCHY
[17:16:05]  Executing: Organization hierarchy
[17:16:08]    Returned 25,435 rows in 3.0s
Loaded 25,435 parent-child relationships


In [17]:
from data_prep import rollup_to_parent

if len(combined_predictions) > 0 and len(hierarchy_df) > 0:
    print("HIERARCHY ROLL-UP")
    print("=" * 50)
    
    combined_predictions, rollup_stats = rollup_to_parent(
        combined_predictions,
        hierarchy_df,
        dim_org_df
    )
    
    print(f"\nRoll-up complete:")
    print(f"  Predictions checked: {rollup_stats['total_checked']:,}")
    print(f"  Rolled up to parent: {rollup_stats['rolled_up']:,}")


HIERARCHY ROLL-UP
[17:16:08]  Hierarchy roll-up: checking for subsidiary matches without location context...
[17:16:08]    Loaded 19,535 child→parent mappings
[17:16:11]    Predictions with missing source location: 5,054
[17:16:11]    Rolled up 689 predictions to parent organizations
[17:16:11]    Sample rollups:
[17:16:11]      Sandoz Group AG (Switzerland) -> Novartis (Switzerland)
[17:16:11]      GlaxoSmithKline (India) -> GlaxoSmithKline (United Kingdom)
[17:16:11]      ERN Skin -> ERN Board of Member States

Roll-up complete:
  Predictions checked: 5,054
  Rolled up to parent: 689


---
## 6. Prediction Statistics


In [18]:
if len(combined_predictions) > 0:
    print("PREDICTION SCORE DISTRIBUTION")
    print("=" * 50)
    
    bins = [0, 0.5, 0.7, 0.85, 0.95, 1.0]
    labels = ['0.5-0.7 (Low)', '0.7-0.85 (Medium)', '0.85-0.95 (High)', '0.95-1.0 (Very High)']
    combined_predictions['confidence_tier'] = pd.cut(
        combined_predictions['match_probability'], 
        bins=bins[1:], 
        labels=labels
    )
    
    print("\nConfidence tiers:")
    tier_counts = combined_predictions['confidence_tier'].value_counts().sort_index()
    for tier, count in tier_counts.items():
        pct = 100 * count / len(combined_predictions)
        print(f"  {tier:<30} | {count:>8,} ({pct:5.1f}%)")


PREDICTION SCORE DISTRIBUTION

Confidence tiers:
  0.5-0.7 (Low)                  |       38 (  0.8%)
  0.7-0.85 (Medium)              |      243 (  4.8%)
  0.85-0.95 (High)               |      116 (  2.3%)
  0.95-1.0 (Very High)           |    4,657 ( 92.1%)


In [19]:
import os 

In [21]:
import openpyxl

In [22]:
# Web-Enhanced LLM Validation with matched_allsci_id
USE_WEB_ENHANCED_VALIDATION = True
config.llm_judge.ENABLE_LLM_VALIDATION = True
MAX_WORKERS = 10
NUM_TO_PROCESS =  5051  # Change to len(combined_predictions) for all

if USE_WEB_ENHANCED_VALIDATION and config.llm_judge.ENABLE_LLM_VALIDATION and len(combined_predictions) > 0:
    from anthropic import Anthropic
    from web_search import get_match_evidence
    from llm_judge import judge_match_with_evidence
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import time
    import pandas as pd
    
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    serper_key = os.environ.get("SERPER_API_KEY")
    
    if not api_key:
        log_step("WARNING: ANTHROPIC_API_KEY not set", "WARN")
    elif not serper_key:
        log_step("WARNING: SERPER_API_KEY not set - web search disabled", "WARN")
    else:
        client = Anthropic(api_key=api_key)
        
        # Limit to NUM_TO_PROCESS
        predictions_to_validate = combined_predictions.head(NUM_TO_PROCESS)
        total = len(predictions_to_validate)
        
        print("\n" + "=" * 70)
        print(f"WEB-ENHANCED LLM VALIDATION ({total} predictions)")
        print("  1. DailyMed NDC lookup for FDA labeler")
        print("  2. Serper search for company context")
        print("  3. LLM validation with evidence")
        print("=" * 70)
        
        log_step(f"Validating {total:,} predictions with {MAX_WORKERS} workers...")
        
        model = config.llm_judge.PRIMARY_MODEL
        evidence_stats = {"dailymed_ndc": 0, "serper_only": 0, "none": 0}
        
        def process_single(idx, row_dict):
            """Process single prediction with rich evidence"""
            source_entity = row_dict.get('source_entity_id', '')
            parts = str(source_entity).split('_') if source_entity else []
            ndc_code = parts[0] if len(parts) > 0 else None
            
            # Extract matched_allsci_id from unique_id_l
            unique_id_l = row_dict.get('unique_id_l', '')
            matched_allsci_id = unique_id_l.replace('dim_', '') if unique_id_l else None
            
            evidence = get_match_evidence(
                source_org=row_dict.get('name_r', ''),
                matched_org=row_dict.get('name_l', ''),
                ndc_code=ndc_code,
                drug_name=row_dict.get('drug_brand_name') or row_dict.get('drug_generic_name')
            )
            
            if evidence.get('drug_label_source') == 'dailymed_ndc':
                source = 'dailymed_ndc'
            elif evidence.get('company_context'):
                source = 'serper_only'
            else:
                source = 'none'
            
            dim_org_record = {
                'name': row_dict.get('name_l'),
                'country_code': row_dict.get('country_code_l')
            }
            
            result = judge_match_with_evidence(row_dict, dim_org_record, evidence, client, model)
            return {
                'source_org': row_dict.get('name_r'),
                'matched_org': row_dict.get('name_l'),
                'matched_allsci_id': matched_allsci_id,
                'fda_labeler': evidence.get('fda_labeler'),
                'fda_drug': evidence.get('fda_drug'),
                'llm_match': result.get('match'),
                'llm_confidence': result.get('confidence'),
                'llm_reason': result.get('reason'),
                'evidence_source': source,
                'ndc_code': ndc_code,
                'prediction_idx': idx
            }
        
        results = []
        completed = 0
        start_time = time.time()
        
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {
                executor.submit(process_single, idx, row.to_dict()): idx
                for idx, row in predictions_to_validate.iterrows()
            }
            
            for future in as_completed(futures):
                try:
                    result = future.result()
                    results.append(result)
                    evidence_stats[result['evidence_source']] = evidence_stats.get(result['evidence_source'], 0) + 1
                except Exception as e:
                    idx = futures[future]
                    row_dict = predictions_to_validate.loc[idx].to_dict()
                    unique_id_l = row_dict.get('unique_id_l', '')
                    results.append({
                        'source_org': row_dict.get('name_r'),
                        'matched_org': row_dict.get('name_l'),
                        'matched_allsci_id': unique_id_l.replace('dim_', '') if unique_id_l else None,
                        'fda_labeler': None,
                        'fda_drug': None,
                        'llm_match': None,
                        'llm_confidence': 0,
                        'llm_reason': f'error: {str(e)[:100]}',
                        'evidence_source': 'none',
                        'ndc_code': None,
                        'prediction_idx': idx
                    })
                    evidence_stats['none'] = evidence_stats.get('none', 0) + 1
                
                completed += 1
                if completed % 10 == 0 or completed == total:
                    elapsed = time.time() - start_time
                    rate = completed / elapsed if elapsed > 0 else 0
                    eta = (total - completed) / rate if rate > 0 else 0
                    print(f"\r  Progress: {completed:,}/{total:,} ({100*completed/total:.1f}%) | "
                          f"Rate: {rate:.1f}/s | ETA: {eta:.0f}s", end="", flush=True)
        
        print()
        
        elapsed_total = time.time() - start_time
        print("\n" + "=" * 70)
        print("EVIDENCE SOURCE STATISTICS")
        print("=" * 70)
        for source, count in sorted(evidence_stats.items(), key=lambda x: -x[1]):
            pct = 100 * count / total
            print(f"  {source:<20} | {count:>5} ({pct:5.1f}%)")
        
        # Save to Excel
        results_df = pd.DataFrame(results)
        excel_path = config.paths.DATA_DIR + f"/web_enhanced_validation_{total}.xlsx"
        results_df.to_excel(excel_path, index=False)
        
        print(f"\n  Total time: {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)")
        print(f"  Saved to: {excel_path}")
        
        # Summary
        matches = sum(1 for r in results if r.get('llm_match') == True)
        rejects = sum(1 for r in results if r.get('llm_match') == False)
        print(f"\n  Matches: {matches}/{total} | Rejects: {rejects}/{total}")
        
        config.llm_judge.ENABLE_LLM_VALIDATION = False
else:
    log_step("Skipping web-enhanced validation")


WEB-ENHANCED LLM VALIDATION (5051 predictions)
  1. DailyMed NDC lookup for FDA labeler
  2. Serper search for company context
  3. LLM validation with evidence
[17:16:22]  Validating 5,051 predictions with 10 workers...
  Progress: 5,051/5,051 (100.0%) | Rate: 1.8/s | ETA: 0sss

EVIDENCE SOURCE STATISTICS
  dailymed_ndc         |  3810 ( 75.4%)
  serper_only          |  1180 ( 23.4%)
  none                 |    61 (  1.2%)

  Total time: 2744.1s (45.7 min)
  Saved to: /Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/data/web_enhanced_validation_5051.xlsx

  Matches: 1384/5051 | Rejects: 3667/5051


In [24]:
import pandas as pd

# Load validation results
validation = pd.read_excel('/Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/data/web_enhanced_validation_5051.xlsx')

# Filter to confirmed matches only
matches = validation[validation['llm_match'] == True]
matched_names = matches['source_org'].unique().tolist()

print(f"Total confirmed matches: {len(matched_names)}")

# Format names for SQL IN clause
names_sql = ", ".join([f"'{name.replace(chr(39), chr(39)+chr(39))}'" for name in matched_names])

# Query cross-source impact for ALL matched names
impact_query = f"""
SELECT 
    source_table,
    COUNT(*) as record_count,
    COUNT(DISTINCT name) as unique_orgs
FROM allsci_prod_gold.potential_mismatched_organizations
WHERE name IN ({names_sql})
GROUP BY source_table
ORDER BY record_count DESC
"""

print("\n" + "=" * 70)
print("CROSS-SOURCE IMPACT - ALL CONFIRMED MATCHES")
print("=" * 70)

impact_df = db.execute_query(impact_query, "Total cross-source impact")
print(impact_df.to_string())

total_records = impact_df['record_count'].sum()
print(f"\n{'='*70}")
print(f"TOTAL RECORDS RESOLVABLE: {total_records:,}")
print(f"UNIQUE ORGANIZATIONS: {len(matched_names)}")
print(f"{'='*70}")

# Break down by source
print("\nBy source table:")
for _, row in impact_df.iterrows():
    pct = 100 * row['record_count'] / total_records
    print(f"  {row['source_table']:<50} | {row['record_count']:>10,} ({pct:5.1f}%) | {row['unique_orgs']:>4} orgs")

Total confirmed matches: 1384

CROSS-SOURCE IMPACT - ALL CONFIRMED MATCHES
[18:02:56]  Executing: Total cross-source impact
[18:03:00]    Returned 9 rows in 4.2s
                                            source_table  record_count  unique_orgs
0                              open_fda_silver.ndc_drugs         71473         1384
1                                   uspto_silver.patents         31751          315
2  nih_clinical_trials_gov_silver.cl_trial_organizations          9019          119
3       nih_clinical_trials_gov_silver.cl_trial_sponsors          8517          127
4            who_clinical_trials_silver.studies_metadata          6304          263
5  nih_clinical_trials_gov_silver.cl_trial_collaborators          4594          132
6      nih_clinical_trials_gov_silver.cl_trial_locations           248           36
7        legacy_alpha_silver._organizations_consolidated            55           55
8          chinese_clinical_trials_silver.trial_contacts            26            

In [25]:
impact_df

,source_table,record_count,unique_orgs
0,open_fda_silver.ndc_drugs,71473,1384
1,uspto_silver.patents,31751,315
2,nih_clinical_trials_gov_silver.cl_trial_organizations,9019,119
3,nih_clinical_trials_gov_silver.cl_trial_sponsors,8517,127
4,who_clinical_trials_silver.studies_metadata,6304,263
5,nih_clinical_trials_gov_silver.cl_trial_collaborators,4594,132
6,nih_clinical_trials_gov_silver.cl_trial_locations,248,36
7,legacy_alpha_silver._organizations_consolidated,55,55
8,chinese_clinical_trials_silver.trial_contacts,26,8


In [27]:
import pandas as pd

# Load your validation results
validation = pd.read_excel('/Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/data/web_enhanced_validation_5051.xlsx')

# Filter to confirmed matches only
matches = validation[validation['llm_match'] == True]
print(f"Your confirmed matches: {len(matches)}")

# Get unique source_org names from your matches
matched_names = matches['source_org'].unique().tolist()
names_sql = ", ".join([f"'{name.replace(chr(39), chr(39)+chr(39))}'" for name in matched_names])

# Query what's already in resolved_mismatched_organizations
already_resolved_query = f"""
SELECT DISTINCT 
    name,
    org_allsci_id,
    source_table,
    COUNT(*) as record_count
FROM allsci_prod_gold.resolved_mismatched_organizations
WHERE name IN ({names_sql})
GROUP BY name, org_allsci_id, source_table
ORDER BY record_count DESC
"""

print("\n" + "=" * 70)
print("ALREADY RESOLVED (in resolved_mismatched_organizations)")
print("=" * 70)
already_resolved_df = db.execute_query(already_resolved_query, "Already resolved")
print(already_resolved_df)

# Compare: which of your matches are new vs already resolved?
already_resolved_names = set(already_resolved_df['name'].unique()) if len(already_resolved_df) > 0 else set()
your_match_names = set(matched_names)

new_to_resolve = your_match_names - already_resolved_names
already_done = your_match_names & already_resolved_names

print("\n" + "=" * 70)
print("COMPARISON SUMMARY")
print("=" * 70)
print(f"Your confirmed matches:     {len(your_match_names)}")
print(f"Already in resolved table:  {len(already_done)}")
print(f"NEW (need to insert):       {len(new_to_resolve)}")

# Show sample of new ones
if new_to_resolve:
    print("\nSample NEW organizations to resolve:")
    new_matches = matches[matches['source_org'].isin(new_to_resolve)][['source_org', 'matched_org', 'matched_allsci_id']].drop_duplicates().head(10)
    print(new_matches.to_string())

# Show already resolved
if already_done:
    print("\nAlready resolved organizations:")
    for name in list(already_done)[:5]:
        print(f"  - {name}")

Your confirmed matches: 1384

ALREADY RESOLVED (in resolved_mismatched_organizations)
[18:04:15]  Executing: Already resolved
[18:04:18]    Returned 15 rows in 2.9s
                             name                        org_allsci_id  \
0              Amerisource Bergen  ASC-OR-0000000117943-1.0-1755116741   
1              Amerisource Bergen  ASC-OR-0000000088377-1.0-1724880260   
2              AMERISOURCE BERGEN  ASC-OR-0000000117943-1.0-1755116741   
3              AMERISOURCE BERGEN  ASC-OR-0000000088377-1.0-1724880260   
4           CHANEL PARFUMS BEAUTE  ASC-OR-0000000056030-1.0-1724880250   
5                          Target  ASC-OR-0000000078308-1.0-1724880256   
6                   THE KROGER CO  ASC-OR-0000000111405-1.0-1740176393   
7              AmeriSource Bergen  ASC-OR-0000000117943-1.0-1755116741   
8              AmeriSource Bergen  ASC-OR-0000000088377-1.0-1724880260   
9                   The Kroger CO  ASC-OR-0000000111405-1.0-1740176393   
10                   

In [30]:
import pandas as pd

validation = pd.read_excel('data/web_enhanced_validation_5051.xlsx')

# Separate matches vs rejections
confirmed = validation[validation['llm_match'] == True]
rejected = validation[validation['llm_match'] == False]

print(f"Confirmed: {len(confirmed)}, Rejected: {len(rejected)}")

# Analyze rejection reasons - look for patterns
print("\nSample rejection reasons:")
for reason in rejected['llm_reason'].head(20):
    print(f"  - {reason[:1000]}")

Confirmed: 1384, Rejected: 3667

Sample rejection reasons:
  - Different companies: Dharma Research Inc. is a US pharmaceutical company making dental products, while Universitas Tri Dharma is an Indonesian university. No corporate relationship indicated.
  - Different company names and business focus - A-S Medication Solutions specializes in pharmaceutical dispensing solutions while Medication Management appears to be a different entity with no evidence of corporate relationship
  - Different company names with distinct business focuses - 'Regenerative Processing Plant, LLC' (pharmaceutical manufacturing) vs 'Regenerative Medical Solutions' (medical solutions provider). No evidence of corporate relationship or alternative naming convention.
  - Old East Main CO is a US-based subsidiary of Dolgencorp/Dollar General used for drug labeling, while Tata Main Hospital is an Indian healthcare institution. These are completely different entities in different countries and industries with no co

In [31]:
# UNMATCHED ORGANIZATIONS IMPACT ANALYSIS
import pandas as pd

# Load unmatched organizations
unmatched = pd.read_parquet('data/openfda_no_match.parquet')
print(f"Total unmatched organizations: {len(unmatched):,}")

# Get their names for cross-source query
unmatched_names = unmatched['name'].unique().tolist()
print(f"Unique unmatched names: {len(unmatched_names):,}")

# Format for SQL
names_sql = ", ".join([f"'{name.replace(chr(39), chr(39)+chr(39))}'" for name in unmatched_names])

# Query frequency in OpenFDA
freq_query = f"""
SELECT name, COUNT(*) as frequency
FROM allsci_prod_gold.potential_mismatched_organizations
WHERE name IN ({names_sql})
  AND source_table = 'open_fda_silver.ndc_drugs'
GROUP BY name
ORDER BY frequency DESC
"""

print("\n" + "=" * 70)
print("UNMATCHED ORGANIZATIONS - OPENFDA FREQUENCY")
print("=" * 70)
unmatched_freq = db.execute_query(freq_query, "Unmatched org frequencies")
print(f"\nTop 20 unmatched by frequency:")
print(unmatched_freq.head(20).to_string())
print(f"\nTotal OpenFDA records from unmatched orgs: {unmatched_freq['frequency'].sum():,}")

# Cross-source impact for unmatched
impact_query = f"""
SELECT 
    source_table,
    COUNT(*) as record_count,
    COUNT(DISTINCT name) as unique_orgs
FROM allsci_prod_gold.potential_mismatched_organizations
WHERE name IN ({names_sql})
GROUP BY source_table
ORDER BY record_count DESC
"""

print("\n" + "=" * 70)
print("CROSS-SOURCE IMPACT - UNMATCHED ORGANIZATIONS")
print("=" * 70)
impact_df = db.execute_query(impact_query, "Unmatched cross-source impact")
print(impact_df.to_string())

total_unmatched_records = impact_df['record_count'].sum()
print(f"\n{'='*70}")
print(f"TOTAL RECORDS IF UNMATCHED WERE RESOLVED: {total_unmatched_records:,}")
print(f"{'='*70}")

# Compare to confirmed matches
print("\n" + "=" * 70)
print("COMPARISON: MATCHED vs UNMATCHED POTENTIAL")
print("=" * 70)
print(f"Confirmed matches:     1,384 orgs -> 131,987 records")
print(f"Unmatched orgs:        {len(unmatched_names):,} orgs -> {total_unmatched_records:,} records (potential)")
print(f"Coverage if fixed:     {131987 + total_unmatched_records:,} total records")

Total unmatched organizations: 4,935
Unique unmatched names: 4,935

UNMATCHED ORGANIZATIONS - OPENFDA FREQUENCY
[18:12:36]  Executing: Unmatched org frequencies
[18:12:45]    Returned 4,935 rows in 8.8s

Top 20 unmatched by frequency:
                                          name  frequency
0                         Bryant Ranch Prepack       9478
1                            REMEDYREPACK INC.       5107
2                             Proficient Rx LP       3676
3                  PD-Rx Pharmaceuticals, Inc.       2456
4                  NuCare Pharmaceuticals,Inc.       1979
5     Aphena Pharma Solutions - Tennessee, LLC       1872
6                           Asclemed USA, Inc.       1364
7                            Kenvue Brands LLC       1243
8                          Uriel Pharmacy Inc.       1223
9   Professional Complementary Health Formulas       1210
10                           Chartwell RX, LLC       1185
11                        Actavis Pharma, Inc.       1174
12         